In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
customers = pd.read_csv('data/raw/olist_customers_dataset.csv')

In [3]:
customers.dtypes

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [4]:
customers['customer_id'] = customers['customer_id'].str.strip()
customers['customer_unique_id'] = customers['customer_unique_id'].str.strip()
customers['customer_city'] = customers['customer_city'].str.strip()
customers['customer_state'] = customers['customer_state'].str.strip()

In [5]:
customers.nunique()

customer_id                 99441
customer_unique_id          96096
customer_zip_code_prefix    14994
customer_city                4119
customer_state                 27
dtype: int64

In [6]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [7]:
customers[customers.duplicated(subset = ['customer_unique_id'],keep = 'first')].count()
# from this output we can see that about 3345 repurchases are made

customer_id                 3345
customer_unique_id          3345
customer_zip_code_prefix    3345
customer_city               3345
customer_state              3345
dtype: int64

In [8]:
# we better convert the the city state as categorical variable and zipcode as string because it has large numbers
# since customer_city is already string we are not making any changes
# we just gonna change the zip_code and customer_state data types

customers['customer_state'] = customers['customer_state'].astype('category')
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype('str')

In [9]:
customers.dtypes

customer_id                      str
customer_unique_id               str
customer_zip_code_prefix         str
customer_city                    str
customer_state              category
dtype: object

In [10]:
orders = pd.read_csv('data/raw/olist_orders_dataset.csv')

In [11]:
orders = orders.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

In [12]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [13]:
orders['order_status'] = orders['order_status'].astype('category')

In [14]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'],format = '%Y-%m-%d %H:%M:%S')
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'],format = '%Y-%m-%d %H:%M:%S')
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'],format = '%Y-%m-%d %H:%M:%S')
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'],format = '%Y-%m-%d %H:%M:%S')
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'],format = '%Y-%m-%d %H:%M:%S')

In [15]:
orders['purchase_year'] = orders['order_purchase_timestamp'].dt.year
orders['purchase_month'] = orders['order_purchase_timestamp'].dt.month
orders['purchase_day'] = orders['order_purchase_timestamp'].dt.day
orders['purchase_day_of_week'] = orders['order_purchase_timestamp'].dt.dayofweek

In [16]:
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days.astype('Int64')
orders['estimated_delivery_days'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.days.astype('Int64')
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date'] ).dt.days.clip(lower = 0).astype('Int64')
orders['delivery_difference'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date'] ).dt.days.astype('Int64')

In [17]:
order_items = pd.read_csv('data/raw/olist_order_items_dataset.csv')

In [18]:
order_items = order_items.apply(lambda col: col.str.strip() if col.dtype == 'object' else col)

In [19]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'],format = '%Y-%m-%d %H:%M:%S')

In [20]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()
# so everything is unique here 
# nothing to worry

np.int64(0)

In [21]:
order_items['order_value'] = order_items['price'] + order_items['freight_value']

In [22]:
order_items['order_item_id'] = order_items['order_item_id'].astype('category')

In [23]:
products = pd.read_csv('data/raw/olist_products_dataset.csv')

In [24]:
products['product_category_name'].nunique()

73

In [25]:
translation = pd.read_csv('data/raw/product_category_name_translation.csv')

In [26]:
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

In [27]:
products['product_category_name'] = products['product_category_name_english']

In [28]:
products.drop(
    columns = ['product_category_name_english'],
    inplace = True
)

In [29]:
products['product_category_name'] = products['product_category_name'].astype('category')

In [30]:
products['product_weight_g'].describe()

count    32949.000000
mean      2276.472488
std       4282.038731
min          0.000000
25%        300.000000
50%        700.000000
75%       1900.000000
max      40425.000000
Name: product_weight_g, dtype: float64

In [31]:
products.loc[products["product_weight_g"].idxmax()]

product_id                    26644690fde745fc4654719c3904e1db
product_category_name                           bed_bath_table
product_name_lenght                                       59.0
product_description_lenght                               534.0
product_photos_qty                                         1.0
product_weight_g                                       40425.0
product_length_cm                                         13.0
product_height_cm                                         65.0
product_width_cm                                          28.0
Name: 25166, dtype: object

In [32]:
filt = (products['product_category_name'] == 'bed_bath_table')
products.loc[filt].describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,3029.000000,3029.000000,3029.000000,3029.000000,3029.000000,3029.000000,3029.000000
mean,51.728293,464.483328,1.392539,2456.405084,36.774183,14.091119,30.589964
std,7.646314,385.235244,0.874992,3732.667484,13.680511,9.628311,11.182182
min,16.000000,58.000000,1.000000,0.000000,12.000000,2.000000,7.000000
25%,49.000000,236.000000,1.000000,700.000000,28.000000,8.000000,22.000000
50%,54.000000,365.000000,1.000000,1250.000000,37.000000,11.000000,30.000000
75%,57.000000,609.000000,1.000000,2300.000000,45.000000,17.000000,37.000000
max,68.000000,3863.000000,9.000000,40425.000000,105.000000,93.000000,118.000000


In [33]:
products.loc[filt].sort_values('product_weight_g').tail(10)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
23664,61779602040b50203b4cdb85dbf2fa83,bed_bath_table,54.0,714.0,2.0,25700.0,45.0,40.0,32.0
18519,24c5c7be57099780f07ded545f898d8b,bed_bath_table,53.0,539.0,1.0,26550.0,58.0,50.0,49.0
5538,5947c33d0250e65638336c25f5a8ab48,bed_bath_table,48.0,354.0,1.0,26600.0,45.0,15.0,35.0
11126,2d2294e3367e0514a129389d9da68670,bed_bath_table,29.0,2113.0,8.0,27050.0,90.0,20.0,90.0
22450,821719b9a2288a792ba3ca0c9962f467,bed_bath_table,58.0,3863.0,5.0,27050.0,90.0,20.0,90.0
20556,97c2f81082289198f18c18dd5d936971,bed_bath_table,40.0,416.0,1.0,27500.0,60.0,52.0,52.0
17765,81258a06e30546c898c31a062ae3da08,bed_bath_table,57.0,258.0,2.0,30000.0,43.0,19.0,35.0
17914,616c497aa9cc2d4ce93eb380cf5cc121,bed_bath_table,54.0,360.0,1.0,30000.0,45.0,15.0,35.0
3213,4abee1df902ca6e48fbe864fce3859bc,bed_bath_table,48.0,610.0,2.0,30000.0,80.0,40.0,70.0
25166,26644690fde745fc4654719c3904e1db,bed_bath_table,59.0,534.0,1.0,40425.0,13.0,65.0,28.0


In [34]:
products.groupby('product_category_name')['product_weight_g'].describe().head(25)

,count,mean,std,min,25%,50%,75%,max
product_category_name,,,,,,,,
agro_industry_and_commerce,74.0,5263.405405,5913.073074,50.0,687.5,2681.5,8950.00,30000.0
air_conditioning,124.0,4459.959677,4775.984418,100.0,437.5,3371.0,6325.00,23300.0
art,55.0,1691.763636,2551.614333,100.0,350.0,700.0,1700.00,15400.0
arts_and_craftmanship,19.0,1164.578947,1599.096978,100.0,258.5,500.0,1225.00,6850.0
audio,58.0,641.637931,923.358452,100.0,250.0,400.0,605.25,6663.0
auto,1900.0,2654.650526,4345.318345,50.0,300.0,900.0,3206.25,30000.0
baby,918.0,3655.201525,5665.129577,50.0,400.0,850.0,5106.25,30000.0
bed_bath_table,3029.0,2456.405084,3732.667484,0.0,700.0,1250.0,2300.00,40425.0
books_general_interest,216.0,746.611111,1094.491160,100.0,350.0,500.0,850.00,13300.0


In [35]:
products.groupby('product_category_name')['product_height_cm'].describe().head(25)

,count,mean,std,min,25%,50%,75%,max
product_category_name,,,,,,,,
agro_industry_and_commerce,74.0,28.945946,21.270529,2.0,13.00,25.5,41.75,105.0
air_conditioning,124.0,23.887097,16.577588,2.0,10.00,18.5,36.00,76.0
art,55.0,11.800000,10.043240,2.0,5.50,10.0,15.50,62.0
arts_and_craftmanship,19.0,9.789474,5.912125,2.0,6.00,9.0,11.00,25.0
audio,58.0,11.500000,6.105074,2.0,7.25,11.0,16.00,31.0
auto,1900.0,16.241579,12.791933,2.0,7.00,14.0,20.00,86.0
baby,918.0,21.617647,16.591138,2.0,10.00,16.0,30.00,97.0
bed_bath_table,3029.0,14.091119,9.628311,2.0,8.00,11.0,17.00,93.0
books_general_interest,216.0,9.773148,8.546422,2.0,3.00,6.0,15.00,40.0


In [36]:
merge_df = (
    customers
    .merge(orders)
    .merge(order_items)
    .merge(products)
)

In [37]:
final_df = merge_df[[
    'customer_unique_id',
    'order_id',
    'customer_state',
    'order_purchase_timestamp',
    'product_category_name',
    'price',
    'freight_value',
    'delivery_days'
]]

In [38]:
final_df['order_value'] = final_df['price'] + final_df['freight_value'] 

In [39]:
final_df.to_csv('data/processed/clean_transactions.csv',index=False)

In [40]:
customer_features = merge_df.groupby('customer_unique_id').agg(
    total_orders= ('order_id', 'nunique'),
    total_items = ('product_id','count'),
    total_spend = ('order_value','sum'),
    average_item_price = ('price','mean'),
    total_freight = ('freight_value','sum'),
    number_of_categories = ('product_category_name','nunique'),
    preferred_category= ("product_category_name", lambda x: x.dropna().mode().iloc[0]
                        if not x.dropna().empty else 'Unknown'),
    customer_state = ('customer_state','first')
).reset_index()

In [41]:
order_totals = (
    merge_df
    .groupby(['customer_unique_id', 'order_id'])['order_value']
    .sum()
    .reset_index()
)

average_order_value = (
    order_totals
    .groupby('customer_unique_id')['order_value']
    .mean()
    .rename('average_order_value')
)

customer_features = customer_features.merge(
    average_order_value,
    on='customer_unique_id',
    how='left'
)

In [42]:
customer_features['first_purchase_date'] = (merge_df.groupby('customer_unique_id')['order_purchase_timestamp'].min().values)
customer_features['last_purchase_date'] = (merge_df.groupby('customer_unique_id')['order_purchase_timestamp'].max().values)

In [43]:
customer_features['customer_lifetime_days'] = (
    customer_features['last_purchase_date']
    - customer_features['first_purchase_date']
).dt.days

In [44]:
customer_features['purchase_frequency'] = (
    customer_features['total_orders'] /
    customer_features['customer_lifetime_days'].replace(0, 1)
)

In [45]:
customer_features['average_delivery_days'] = (merge_df.groupby('customer_unique_id')['delivery_days'].mean().values)
customer_features['average_delivery_delay'] = (merge_df.groupby('customer_unique_id')['delivery_delay_days'].mean().values)

In [46]:
late_rate = (
    merge_df.groupby("customer_unique_id")["delivery_difference"]
    .apply(lambda x: (x.dropna() > 0).mean())
    .rename("late_order_rate")
)

customer_features = customer_features.merge(
    late_rate,
    on="customer_unique_id",
    how="left"
)

In [47]:
customer_features.to_csv('data/processed/customer_features.csv',index = False)